# 02 — MVTec: HPO log preprocessing

Loads the raw Optuna SQLite log from notebook 01, filters to the 9,976 COMPLETE trials,
selects and types the 13 hard/soft configuration columns plus the validation-F1 target, and
writes the surrogate-ready table (`surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv`)
consumed by every downstream MVTec notebook. Implements Section 3.3 of the paper.

Also produces the preprocessing diagnostics (F1 distribution, best-so-far curve,
F1-by-hyperparameter boxplots) under `figures_preprocessing/` and `tables_preprocessing/`.


In [ ]:
from pathlib import Path
import json
import sqlite3

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

# ------------------------------------------------------------
# Robust project/root detection
# ------------------------------------------------------------

CWD = Path.cwd()
print("Current working directory:")
print(CWD)

DB_FILENAME = "mvtec_4class_patchcore_hpo_logs.db"

candidate_paths = []

# Common locations depending on where the notebook is opened from
candidate_paths.extend([
    CWD / DB_FILENAME,
    CWD / "outputs_patchcore" / DB_FILENAME,
    CWD / "hpo_mvtec_4class" / "outputs_patchcore" / DB_FILENAME,
    CWD.parent / "outputs_patchcore" / DB_FILENAME,
    CWD.parent / "hpo_mvtec_4class" / "outputs_patchcore" / DB_FILENAME,
])

# Recursive search from current directory and parent, limited to this filename
for base in [CWD, CWD.parent]:
    try:
        candidate_paths.extend(base.rglob(DB_FILENAME))
    except Exception:
        pass

# Remove duplicates while preserving order
seen = set()
candidate_paths_unique = []
for p in candidate_paths:
    p = p.resolve()
    if p not in seen:
        seen.add(p)
        candidate_paths_unique.append(p)

print("\nCandidate DB paths:")
db_candidates_existing = []

for p in candidate_paths_unique:
    exists = p.exists()
    size_mb = p.stat().st_size / (1024 * 1024) if exists else 0
    print(f"- exists={exists:<5} size={size_mb:8.3f} MB | {p}")
    if exists:
        db_candidates_existing.append(p)

if not db_candidates_existing:
    raise FileNotFoundError(
        "Could not find mvtec_4class_patchcore_hpo_logs.db. "
        "Check where the notebook is running from using Path.cwd()."
    )

# Pick the largest existing DB by default.
# This avoids accidentally using an empty DB created by a wrong path.
DB_PATH = max(db_candidates_existing, key=lambda p: p.stat().st_size)

print("\nSelected DB:")
print(DB_PATH)
print("Size MB:", DB_PATH.stat().st_size / (1024 * 1024))

# Define project root from the selected DB location
OUTPUTS_DIR = DB_PATH.parent
ROOT = OUTPUTS_DIR.parent

SURROGATE_DIR = ROOT / "surrogate_ready_dataset"
FIG_DIR = ROOT / "figures_preprocessing"
TABLE_DIR = ROOT / "tables_preprocessing"

SURROGATE_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

BEST_PARAMS_PATH = OUTPUTS_DIR / "best_params.json"
BEST_RESULTS_PATH = OUTPUTS_DIR / "best_params_final_results.csv"

print("\nResolved paths:")
print("ROOT:", ROOT)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("SURROGATE_DIR:", SURROGATE_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)

print("\nFile checks:")
print("DB exists:", DB_PATH.exists(), DB_PATH)
print("Best params JSON exists:", BEST_PARAMS_PATH.exists(), BEST_PARAMS_PATH)
print("Best final results CSV exists:", BEST_RESULTS_PATH.exists(), BEST_RESULTS_PATH)

In [ ]:
# ------------------------------------------------------------
# Inspect SQLite content before loading with Optuna
# ------------------------------------------------------------

def inspect_sqlite_db(db_path):
    with sqlite3.connect(db_path) as conn:
        tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
            conn
        )

        print("SQLite tables:")
        display(tables)

        if "studies" in tables["name"].tolist():
            studies_sql = pd.read_sql_query("SELECT * FROM studies;", conn)
            print("\nStudies table:")
            display(studies_sql)
        else:
            print("\nWARNING: No 'studies' table found. This may not be an Optuna RDB database.")

inspect_sqlite_db(DB_PATH)

# Use absolute path for Optuna storage
STORAGE = f"sqlite:///{DB_PATH.resolve().as_posix()}"

summaries = optuna.get_all_study_summaries(storage=STORAGE)

print("\nOptuna studies found:", len(summaries))

for i, s in enumerate(summaries):
    best_value = s.best_trial.value if s.best_trial is not None else None
    print(
        f"{i}: name={s.study_name} | trials={s.n_trials} | "
        f"direction={s.direction} | best={best_value}"
    )

if len(summaries) == 0:
    raise ValueError(
        "Optuna found 0 studies in the selected database. "
        "Check the SQLite tables printed above. If the selected DB is very small, "
        "you may have accidentally created an empty DB from the earlier wrong path."
    )

In [ ]:
STUDY_NAME = summaries[0].study_name

study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=STORAGE
)

print("Loaded study:", study.study_name)
print("Number of trials:", len(study.trials))
print("Best value:", study.best_value)
print("Best trial:", study.best_trial.number)

In [ ]:
df_raw = study.trials_dataframe(
    attrs=("number", "value", "state", "params", "user_attrs")
)

print("Raw trials dataframe:", df_raw.shape)
display(df_raw.head())
display(df_raw.head().T)

In [ ]:
df = df_raw[df_raw["state"] == "COMPLETE"].copy()

df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["value"]).reset_index(drop=True)

print("Complete trials with valid value:", df.shape)
print("Best validation F1:", df["value"].max())
print("Best trial number:", int(df.loc[df["value"].idxmax(), "number"]))

display(
    df["value"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
param_cols = [c for c in df.columns if c.startswith("params_")]
user_attr_cols = [c for c in df.columns if c.startswith("user_attrs_")]

print("Parameter columns:")
for c in param_cols:
    print("-", c)

print("\nUser attribute columns:")
for c in user_attr_cols:
    print("-", c)

In [ ]:
best_row = df.loc[df["value"].idxmax()].copy()

print("Best trial:", int(best_row["number"]))
print("Best validation F1:", float(best_row["value"]))

best_params = {
    k.replace("params_", ""): best_row[k]
    for k in df.columns
    if k.startswith("params_")
}

best_attrs = {
    k.replace("user_attrs_", ""): best_row[k]
    for k in df.columns
    if k.startswith("user_attrs_")
}

print("\nBest params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

print("\nBest user attributes:")
for k, v in best_attrs.items():
    print(f"  {k}: {v}")

In [ ]:
if BEST_PARAMS_PATH.exists():
    with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
        best_params_json = json.load(f)

    print("best_params.json:")
    print(json.dumps(best_params_json, indent=2)[:5000])
else:
    print("No best_params.json found.")

if BEST_RESULTS_PATH.exists():
    best_results = pd.read_csv(BEST_RESULTS_PATH)
    print("\nbest_params_final_results.csv:", best_results.shape)
    display(best_results.head())
else:
    print("No best_params_final_results.csv found.")

In [ ]:
# Validation F1 distribution
plt.figure(figsize=(7, 4))
plt.hist(df["value"], bins=60)
plt.title("MVTec 4-class HPO: validation F1 distribution")
plt.xlabel("Validation F1")
plt.ylabel("Count")
plt.tight_layout()

plt.savefig(FIG_DIR / "mvtec_f1_distribution.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "mvtec_f1_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


# Best validation F1 over trials
df_sorted = df.sort_values("number").copy()
df_sorted["best_so_far"] = df_sorted["value"].cummax()

plt.figure(figsize=(7, 4))
plt.plot(df_sorted["number"], df_sorted["best_so_far"])
plt.title("MVTec 4-class HPO: best validation F1 so far")
plt.xlabel("Trial")
plt.ylabel("Best validation F1 so far")
plt.tight_layout()

plt.savefig(FIG_DIR / "mvtec_best_f1_so_far.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "mvtec_best_f1_so_far.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
params_to_plot = [
    "params_backbone",
    "params_layers_key",
    "params_pre_trained",
    "params_image_size_key",
    "params_center_crop_key",
    "params_reduction",
    "params_batch_size",
    "params_max_patches_per_image",
    "params_soft_corruption_level",
]

for col in params_to_plot:
    if col not in df.columns:
        print("Skipping missing column:", col)
        continue

    g = df[[col, "value"]].dropna().copy()
    g[col] = g[col].astype(str)

    if g[col].nunique() > 20:
        print("Skipping high-cardinality column:", col, g[col].nunique())
        continue

    cats = sorted(g[col].unique().tolist())
    data = [g.loc[g[col] == c, "value"].values for c in cats]

    plt.figure(figsize=(8, 4))
    plt.boxplot(data, labels=cats, showfliers=False)
    plt.title(f"MVTec validation F1 by {col.replace('params_', '')}")
    plt.ylabel("Validation F1")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    safe_name = col.replace("params_", "").replace("/", "_")
    plt.savefig(FIG_DIR / f"mvtec_f1_by_{safe_name}.pdf", bbox_inches="tight")
    plt.savefig(FIG_DIR / f"mvtec_f1_by_{safe_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
cols_to_drop = [
    "number",
    "state",
    "datetime_start",
    "datetime_complete",
    "duration",
    "system_attrs_completed_rung_0",
    "system_attrs_completed_rung_1",
    "system_attrs_completed_rung_2",
    "system_attrs_completed_rung_3",
    "user_attrs_f1_val_at_budget",
    "user_attrs_soft_train_fraction_q01",
]

df_clean = df.drop(columns=cols_to_drop, errors="ignore").copy()

print("Shape after dropping metadata columns:", df_clean.shape)
display(df_clean.head().T)

In [ ]:
rounding_rules = {
    "params_soft_train_fraction": 2,
    "params_coreset_sampling_ratio": 3,
    "params_soft_review_budget": 3,
}

for col, ndigits in rounding_rules.items():
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").round(ndigits)

display(df_clean.head().T)

In [ ]:
TARGET = "value"

FEATURES = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_coreset_sampling_ratio",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_num_neighbors",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

keep_cols = [TARGET] + FEATURES

missing_cols = [c for c in keep_cols if c not in df_clean.columns]
available_cols = [c for c in keep_cols if c in df_clean.columns]

print("Expected columns:", len(keep_cols))
print("Available columns:", len(available_cols))
print("Missing columns:", missing_cols)

if missing_cols:
    print("\nAvailable parameter columns:")
    for c in sorted([c for c in df_clean.columns if c.startswith("params_")]):
        print("-", c)

In [ ]:
if missing_cols:
    raise ValueError(
        "Some expected surrogate columns are missing. "
        "Check Cell 13 output and update FEATURES to match the MVTec HPO log."
    )

df_export = df_clean.loc[:, keep_cols].copy()

before = len(df_export)
df_export = df_export.dropna(subset=keep_cols).reset_index(drop=True)
after = len(df_export)

print("Rows before dropna:", before)
print("Rows after dropna:", after)
print("Dropped rows:", before - after)

display(df_export.head())
display(df_export.describe(include="all").T)

In [ ]:
cat_as_str = [
    "params_backbone",
    "params_center_crop_key",
    "params_image_size_key",
    "params_layers_key",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
    "params_batch_size",
    "params_max_patches_per_image",
]

for c in cat_as_str:
    if c in df_export.columns:
        df_export[c] = df_export[c].astype(str)

num_cols = [
    "params_coreset_sampling_ratio",
    "params_num_neighbors",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    TARGET,
]

for c in num_cols:
    if c in df_export.columns:
        df_export[c] = pd.to_numeric(df_export[c], errors="coerce")

before = len(df_export)
df_export = df_export.dropna(subset=num_cols).reset_index(drop=True)
after = len(df_export)

print("Rows before numeric cleaning:", before)
print("Rows after numeric cleaning:", after)
print("Dropped rows:", before - after)

print("\nDtypes:")
display(df_export.dtypes)

display(df_export.head())

In [ ]:
print("Target summary:")
display(df_export[TARGET].describe())

print("\nCategorical domains:")
for c in cat_as_str:
    if c in df_export.columns:
        vals = sorted(df_export[c].dropna().astype(str).unique().tolist())
        print(f"{c}: {vals}")

print("\nNumeric ranges:")
for c in num_cols:
    if c in df_export.columns:
        print(
            f"{c}: min={df_export[c].min():.6f}, "
            f"max={df_export[c].max():.6f}, "
            f"mean={df_export[c].mean():.6f}"
        )

In [ ]:
out_path = SURROGATE_DIR / "mvtec_4class_patchcore_surrogate_dataset.csv"

df_export.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", df_export.shape)

display(df_export.head())

In [ ]:
summary = {
    "study_name": study.study_name,
    "n_trials_total": len(study.trials),
    "n_complete_valid_trials": len(df),
    "n_surrogate_rows": len(df_export),
    "n_features": len(FEATURES),
    "best_validation_f1": float(df["value"].max()),
    "mean_validation_f1": float(df["value"].mean()),
    "median_validation_f1": float(df["value"].median()),
    "min_validation_f1": float(df["value"].min()),
    "max_validation_f1": float(df["value"].max()),
}

summary_df = pd.DataFrame([summary])

summary_path = TABLE_DIR / "mvtec_preprocessing_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df.T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df["value"], bins=60)
axes[0].set_title("Validation F1 distribution")
axes[0].set_xlabel("Validation F1")
axes[0].set_ylabel("Count")

df_sorted = df.sort_values("number").copy()
df_sorted["best_so_far"] = df_sorted["value"].cummax()

axes[1].plot(df_sorted["number"], df_sorted["best_so_far"])
axes[1].set_title("Best validation F1 so far")
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Best validation F1")

fig.suptitle("MVTec 4-class PatchCore HPO log")
plt.tight_layout()

fig_path_pdf = FIG_DIR / "mvtec_hpo_log_summary.pdf"
fig_path_png = FIG_DIR / "mvtec_hpo_log_summary.png"

fig.savefig(fig_path_pdf, bbox_inches="tight")
fig.savefig(fig_path_png, dpi=300, bbox_inches="tight")

print("Saved:", fig_path_pdf)
print("Saved:", fig_path_png)

plt.show()